In [1]:
import duckdb
import os

con = duckdb.connect()

# =============================================================================
# Paths
# =============================================================================
PROCESSED_DIR = r"data\processed"

earnings_prices_clean_file = os.path.join(PROCESSED_DIR, "earnings_prices_clean.parquet")
name_file = os.path.join(PROCESSED_DIR, "security_name_history_small.parquet")
sec_price_file = os.path.join(PROCESSED_DIR, "security_prices_small.parquet")
opt_file = os.path.join(PROCESSED_DIR, "option_prices_small.parquet")

# optional final output
earnings_straddles_out = os.path.join(PROCESSED_DIR, "earnings_straddles_stage1.parquet")

# =============================================================================
# Helper
# =============================================================================
def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

# =============================================================================
# STEP 1: Load clean earnings price sample
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_prices_clean AS
SELECT *
FROM read_parquet('{earnings_prices_clean_file}')
""")

# =============================================================================
# STEP 2: Build CRSP -> OptionMetrics secid map again
# using ncusip on prev_date and latest effect_date <= prev_date
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp_id_history AS
SELECT DISTINCT
    permno,
    date,
    NCUSIP AS ncusip,
    TICKER AS crsp_ticker
FROM read_parquet('data\\processed\\crsp_daily_stock_small_v2.parquet')
WHERE NCUSIP IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_crsp_id AS
SELECT
    e.*,
    c.ncusip,
    c.crsp_ticker
FROM earnings_prices_clean e
LEFT JOIN crsp_id_history c
  ON e.permno = c.permno
 AND e.prev_date = c.date
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_name_history AS
SELECT
    secid,
    ticker AS om_ticker,
    cusip AS om_cusip,
    effect_date
FROM read_parquet('{name_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_secid AS
SELECT *
FROM (
    SELECT
        e.*,
        o.secid,
        o.om_ticker,
        o.om_cusip,
        o.effect_date,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY o.effect_date DESC, o.secid
        ) AS rn
    FROM event_crsp_id e
    LEFT JOIN om_name_history o
      ON e.ncusip = o.om_cusip
     AND o.effect_date <= e.prev_date
) t
WHERE rn = 1
  AND secid IS NOT NULL
""")

# =============================================================================
# STEP 3: Attach OptionMetrics underlying stock price on prev_date
# This is the stock price used for ATM strike selection and implied move scaling
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_sec_prices AS
SELECT
    secid,
    date,
    close AS om_close,
    open AS om_open,
    ticker AS om_price_ticker,
    cusip AS om_price_cusip
FROM read_parquet('{sec_price_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_with_om_price AS
SELECT
    e.*,
    s.om_close,
    s.om_open,
    s.om_price_ticker,
    s.om_price_cusip
FROM event_secid e
LEFT JOIN om_sec_prices s
  ON e.secid = s.secid
 AND e.prev_date = s.date
""")

# =============================================================================
# STEP 4: Keep only rows with valid OptionMetrics underlying price
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE event_ready AS
SELECT *
FROM event_with_om_price
WHERE om_close IS NOT NULL
""")

# =============================================================================
# STEP 5: Find nearest expiry before earnings and nearest expiry after earnings
# based on available option chains on prev_date for the mapped secid
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE option_chain_dates AS
SELECT DISTINCT
    secid,
    date,
    exdate
FROM read_parquet('{opt_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_expiry_before AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.rdq,
        e.secid,
        e.prev_date,
        c.exdate AS before_exdate,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.exdate DESC
        ) AS rn
    FROM event_ready e
    LEFT JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
     AND c.exdate <= e.rdq
) t
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_expiry_after AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.rdq,
        e.secid,
        e.prev_date,
        c.exdate AS after_exdate,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.exdate ASC
        ) AS rn
    FROM event_ready e
    LEFT JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
     AND c.exdate > e.rdq
) t
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_expiries AS
SELECT
    e.*,
    b.before_exdate,
    a.after_exdate
FROM event_ready e
LEFT JOIN event_expiry_before b
  ON e.gvkey = b.gvkey AND e.rdq = b.rdq
LEFT JOIN event_expiry_after a
  ON e.gvkey = a.gvkey AND e.rdq = a.rdq
""")

# =============================================================================
# STEP 6: Keep only events with both before and after expiries
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE event_expiries_clean AS
SELECT *
FROM event_expiries
WHERE before_exdate IS NOT NULL
  AND after_exdate IS NOT NULL
""")

# =============================================================================
# STEP 7: For the AFTER expiry chain, choose ATM strike using |strike - om_close|
# strike_price in OptionMetrics is scaled by 1000
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE after_chain AS
SELECT
    e.gvkey,
    e.rdq,
    e.tic,
    e.permno,
    e.secid,
    e.prev_date,
    e.before_exdate,
    e.after_exdate,
    e.om_close,
    o.exdate,
    o.strike_price,
    o.strike_price / 1000.0 AS strike,
    ABS(o.strike_price / 1000.0 - e.om_close) AS strike_distance
FROM event_expiries_clean e
JOIN read_parquet('{opt_file}') o
  ON e.secid = o.secid
 AND e.prev_date = o.date
 AND e.after_exdate = o.exdate
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE atm_strike_choice AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY gvkey, rdq
            ORDER BY strike_distance ASC, strike ASC
        ) AS rn
    FROM after_chain
) t
WHERE rn = 1
""")

# =============================================================================
# STEP 8: Pull call/put mids for AFTER expiry at chosen strike
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE after_legs AS
SELECT
    a.gvkey,
    a.rdq,
    a.tic,
    a.permno,
    a.secid,
    a.prev_date,
    a.before_exdate,
    a.after_exdate,
    a.om_close,
    a.strike,
    a.strike_price,

    MAX(CASE WHEN o.cp_flag = 'C' THEN o.mid END) AS after_call_mid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.mid END) AS after_put_mid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_bid END) AS after_call_bid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_offer END) AS after_call_ask,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_bid END) AS after_put_bid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_offer END) AS after_put_ask
FROM atm_strike_choice a
LEFT JOIN read_parquet('{opt_file}') o
  ON a.secid = o.secid
 AND a.prev_date = o.date
 AND a.after_exdate = o.exdate
 AND a.strike_price = o.strike_price
GROUP BY
    a.gvkey, a.rdq, a.tic, a.permno, a.secid, a.prev_date,
    a.before_exdate, a.after_exdate, a.om_close, a.strike, a.strike_price
""")

# =============================================================================
# STEP 9: Pull call/put mids for BEFORE expiry at same chosen strike
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE before_legs AS
SELECT
    a.gvkey,
    a.rdq,

    MAX(CASE WHEN o.cp_flag = 'C' THEN o.mid END) AS before_call_mid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.mid END) AS before_put_mid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_bid END) AS before_call_bid,
    MAX(CASE WHEN o.cp_flag = 'C' THEN o.best_offer END) AS before_call_ask,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_bid END) AS before_put_bid,
    MAX(CASE WHEN o.cp_flag = 'P' THEN o.best_offer END) AS before_put_ask
FROM atm_strike_choice a
LEFT JOIN read_parquet('{opt_file}') o
  ON a.secid = o.secid
 AND a.prev_date = o.date
 AND a.before_exdate = o.exdate
 AND a.strike_price = o.strike_price
GROUP BY
    a.gvkey, a.rdq
""")

# =============================================================================
# STEP 10: Combine into stage-1 straddle table
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE earnings_straddles_stage1 AS
SELECT
    a.gvkey,
    a.tic,
    a.permno,
    a.secid,
    a.rdq,
    a.prev_date,
    a.before_exdate,
    a.after_exdate,
    a.om_close,
    a.strike,
    a.strike_price,

    b.before_call_mid,
    b.before_put_mid,
    (b.before_call_mid + b.before_put_mid) AS before_straddle_mid,

    af.after_call_mid,
    af.after_put_mid,
    (af.after_call_mid + af.after_put_mid) AS after_straddle_mid,

    af.after_call_bid,
    af.after_call_ask,
    af.after_put_bid,
    af.after_put_ask,
    b.before_call_bid,
    b.before_call_ask,
    b.before_put_bid,
    b.before_put_ask,

    CASE
        WHEN a.om_close IS NOT NULL AND a.om_close <> 0
             AND (af.after_call_mid + af.after_put_mid) IS NOT NULL
             AND (b.before_call_mid + b.before_put_mid) IS NOT NULL
        THEN ((af.after_call_mid + af.after_put_mid) - (b.before_call_mid + b.before_put_mid)) / a.om_close
        ELSE NULL
    END AS implied_move
FROM atm_strike_choice a
LEFT JOIN after_legs af
  ON a.gvkey = af.gvkey AND a.rdq = af.rdq
LEFT JOIN before_legs b
  ON a.gvkey = b.gvkey AND a.rdq = b.rdq
""")

# =============================================================================
# STEP 11: Save stage-1 table
# =============================================================================
con.execute(f"""
COPY earnings_straddles_stage1 TO '{earnings_straddles_out}' (FORMAT PARQUET)
""")

print(f"\nSaved stage-1 straddle table to:\n{earnings_straddles_out}")

# =============================================================================
# CHECKS
# =============================================================================
show(
    "1. Clean earnings price rows",
    "SELECT COUNT(*) AS n_rows FROM earnings_prices_clean"
)

show(
    "2. Rows with mapped secid",
    "SELECT COUNT(*) AS n_rows FROM event_secid"
)

show(
    "3. Rows with OptionMetrics underlying price on prev_date",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN om_close IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_om_close
    FROM event_with_om_price
    """
)

show(
    "4. Rows with both before and after expiries",
    """
    SELECT
        COUNT(*) AS total_rows
    FROM event_expiries_clean
    """
)

show(
    "5. ATM strike sample",
    """
    SELECT
        tic,
        rdq,
        secid,
        prev_date,
        om_close,
        after_exdate,
        strike,
        strike_distance
    FROM atm_strike_choice
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "6. Straddle completeness",
    """
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN before_straddle_mid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_before_straddle,
        SUM(CASE WHEN after_straddle_mid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_after_straddle,
        SUM(CASE WHEN implied_move IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_implied_move
    FROM earnings_straddles_stage1
    """
)

show(
    "7. Sample stage-1 straddles",
    """
    SELECT
        tic,
        rdq,
        prev_date,
        before_exdate,
        after_exdate,
        om_close,
        strike,
        before_straddle_mid,
        after_straddle_mid,
        implied_move
    FROM earnings_straddles_stage1
    WHERE implied_move IS NOT NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)

show(
    "8. Rows missing before or after straddle",
    """
    SELECT
        tic,
        rdq,
        prev_date,
        before_exdate,
        after_exdate,
        om_close,
        strike,
        before_call_mid,
        before_put_mid,
        after_call_mid,
        after_put_mid
    FROM earnings_straddles_stage1
    WHERE before_straddle_mid IS NULL OR after_straddle_mid IS NULL
    ORDER BY rdq, tic
    LIMIT 30
    """
)


Saved stage-1 straddle table to:
data\processed\earnings_straddles_stage1.parquet

1. Clean earnings price rows
 n_rows
  12776

2. Rows with mapped secid
 n_rows
  12456

3. Rows with OptionMetrics underlying price on prev_date
 total_rows  rows_with_om_close
      12456             12417.0

4. Rows with both before and after expiries
 total_rows
          0

5. ATM strike sample
Empty DataFrame
Columns: [tic, rdq, secid, prev_date, om_close, after_exdate, strike, strike_distance]
Index: []

6. Straddle completeness
 total_rows  rows_with_before_straddle  rows_with_after_straddle  rows_with_implied_move
          0                        NaN                       NaN                     NaN

7. Sample stage-1 straddles
Empty DataFrame
Columns: [tic, rdq, prev_date, before_exdate, after_exdate, om_close, strike, before_straddle_mid, after_straddle_mid, implied_move]
Index: []

8. Rows missing before or after straddle
Empty DataFrame
Columns: [tic, rdq, prev_date, before_exdate, after_

In [2]:
import duckdb
import os

con = duckdb.connect()

PROCESSED_DIR = r"data\processed"

earnings_prices_clean_file = os.path.join(PROCESSED_DIR, "earnings_prices_clean.parquet")
name_file = os.path.join(PROCESSED_DIR, "security_name_history_small.parquet")
sec_price_file = os.path.join(PROCESSED_DIR, "security_prices_small.parquet")
opt_file = os.path.join(PROCESSED_DIR, "option_prices_small.parquet")
crsp_file = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small_v2.parquet")

def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

# -----------------------------------------------------------------------------
# rebuild mapped event sample quickly
# -----------------------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_prices_clean AS
SELECT *
FROM read_parquet('{earnings_prices_clean_file}')
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp_id_history AS
SELECT DISTINCT
    permno,
    date,
    NCUSIP AS ncusip,
    TICKER AS crsp_ticker
FROM read_parquet('{crsp_file}')
WHERE NCUSIP IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_crsp_id AS
SELECT
    e.*,
    c.ncusip,
    c.crsp_ticker
FROM earnings_prices_clean e
LEFT JOIN crsp_id_history c
  ON e.permno = c.permno
 AND e.prev_date = c.date
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_name_history AS
SELECT
    secid,
    ticker AS om_ticker,
    cusip AS om_cusip,
    effect_date
FROM read_parquet('{name_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_secid AS
SELECT *
FROM (
    SELECT
        e.*,
        o.secid,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY o.effect_date DESC, o.secid
        ) AS rn
    FROM event_crsp_id e
    LEFT JOIN om_name_history o
      ON e.ncusip = o.om_cusip
     AND o.effect_date <= e.prev_date
) t
WHERE rn = 1
  AND secid IS NOT NULL
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_sec_prices AS
SELECT
    secid,
    date,
    close AS om_close
FROM read_parquet('{sec_price_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_ready AS
SELECT
    e.*,
    s.om_close
FROM event_secid e
LEFT JOIN om_sec_prices s
  ON e.secid = s.secid
 AND e.prev_date = s.date
WHERE s.om_close IS NOT NULL
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE option_chain_dates AS
SELECT DISTINCT
    secid,
    date,
    exdate
FROM read_parquet('{opt_file}')
""")

# -----------------------------------------------------------------------------
# inspect sample expiries available on prev_date
# -----------------------------------------------------------------------------
show(
    "1. Sample available expiries on prev_date for first few events",
    """
    SELECT
        e.tic,
        e.rdq,
        e.prev_date,
        c.exdate
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    ORDER BY e.rdq, e.tic, c.exdate
    LIMIT 100
    """
)

# -----------------------------------------------------------------------------
# compare counts under different split rules
# -----------------------------------------------------------------------------
show(
    "2. Events with at least one expiry <= prev_date",
    """
    SELECT
        COUNT(DISTINCT e.gvkey || '|' || CAST(e.rdq AS VARCHAR)) AS n_events
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate <= e.prev_date
    """
)

show(
    "3. Events with at least one expiry > prev_date",
    """
    SELECT
        COUNT(DISTINCT e.gvkey || '|' || CAST(e.rdq AS VARCHAR)) AS n_events
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate > e.prev_date
    """
)

show(
    "4. Events with at least one expiry <= rdq",
    """
    SELECT
        COUNT(DISTINCT e.gvkey || '|' || CAST(e.rdq AS VARCHAR)) AS n_events
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate <= e.rdq
    """
)

show(
    "5. Events with at least one expiry > rdq",
    """
    SELECT
        COUNT(DISTINCT e.gvkey || '|' || CAST(e.rdq AS VARCHAR)) AS n_events
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate > e.rdq
    """
)

# -----------------------------------------------------------------------------
# closest expiry before/after prev_date
# -----------------------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TEMP TABLE expiry_before_prev AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.rdq,
        c.exdate,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.exdate DESC
        ) AS rn
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate <= e.prev_date
) t
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE expiry_after_prev AS
SELECT *
FROM (
    SELECT
        e.gvkey,
        e.rdq,
        c.exdate,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY c.exdate ASC
        ) AS rn
    FROM event_ready e
    JOIN option_chain_dates c
      ON e.secid = c.secid
     AND e.prev_date = c.date
    WHERE c.exdate > e.prev_date
) t
WHERE rn = 1
""")

show(
    "6. Events with both before/after expiries around prev_date",
    """
    SELECT
        COUNT(*) AS n_events
    FROM event_ready e
    JOIN expiry_before_prev b
      ON e.gvkey = b.gvkey AND e.rdq = b.rdq
    JOIN expiry_after_prev a
      ON e.gvkey = a.gvkey AND e.rdq = a.rdq
    """
)

show(
    "7. Sample before/after expiries around prev_date",
    """
    SELECT
        e.tic,
        e.rdq,
        e.prev_date,
        b.exdate AS before_exdate,
        a.exdate AS after_exdate
    FROM event_ready e
    JOIN expiry_before_prev b
      ON e.gvkey = b.gvkey AND e.rdq = b.rdq
    JOIN expiry_after_prev a
      ON e.gvkey = a.gvkey AND e.rdq = a.rdq
    ORDER BY e.rdq, e.tic
    LIMIT 50
    """
)

# -----------------------------------------------------------------------------
# nearest two expiries after prev_date
# useful if no expiry on or before prev_date
# -----------------------------------------------------------------------------
show(
    "8. First two expiries after prev_date for sample events",
    """
    SELECT *
    FROM (
        SELECT
            e.tic,
            e.rdq,
            e.prev_date,
            c.exdate,
            ROW_NUMBER() OVER (
                PARTITION BY e.gvkey, e.rdq
                ORDER BY c.exdate ASC
            ) AS rn
        FROM event_ready e
        JOIN option_chain_dates c
          ON e.secid = c.secid
         AND e.prev_date = c.date
        WHERE c.exdate > e.prev_date
    ) t
    WHERE rn <= 2
    ORDER BY rdq, tic, rn
    LIMIT 100
    """
)


1. Sample available expiries on prev_date for first few events
 tic        rdq  prev_date     exdate
NVDA 2018-02-08 2018-02-07 2018-02-16
NVDA 2018-02-08 2018-02-07 2018-02-23
NVDA 2018-02-08 2018-02-07 2018-03-02
NVDA 2018-02-08 2018-02-07 2018-03-09
NVDA 2018-02-08 2018-02-07 2018-03-16
NVDA 2018-02-08 2018-02-07 2018-03-23
NVDA 2018-02-08 2018-02-07 2018-04-20
   A 2018-02-14 2018-02-13 2018-03-16
   A 2018-02-14 2018-02-13 2018-05-18
AMAT 2018-02-14 2018-02-13 2018-02-23
AMAT 2018-02-14 2018-02-13 2018-03-02
AMAT 2018-02-14 2018-02-13 2018-03-09
AMAT 2018-02-14 2018-02-13 2018-03-16
AMAT 2018-02-14 2018-02-13 2018-03-23
AMAT 2018-02-14 2018-02-13 2018-03-29
AMAT 2018-02-14 2018-02-13 2018-04-20
CSCO 2018-02-14 2018-02-13 2018-02-23
CSCO 2018-02-14 2018-02-13 2018-03-02
CSCO 2018-02-14 2018-02-13 2018-03-09
CSCO 2018-02-14 2018-02-13 2018-03-16

[showing first 20 rows out of 100]

2. Events with at least one expiry <= prev_date
 n_events
        0

3. Events with at least one expi

In [3]:
import duckdb
import os

con = duckdb.connect()

# =============================================================================
# PATHS
# =============================================================================
PROCESSED_DIR = r"data\processed"
RAW_DIR = "data"   # change if needed

fundq_file = os.path.join(PROCESSED_DIR, "compustat_fundq_small.parquet")
ccm_file = os.path.join(PROCESSED_DIR, "ccm_link_table_small.parquet")
crsp_file = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small_v2.parquet")
name_file = os.path.join(PROCESSED_DIR, "security_name_history_small.parquet")
sec_price_file = os.path.join(PROCESSED_DIR, "security_prices_small.parquet")
earnings_prices_clean_file = os.path.join(PROCESSED_DIR, "earnings_prices_clean.parquet")

# RAW OPTIONMETRICS OPTION PRICES FILE
raw_opt_file = os.path.join(RAW_DIR, "option_prices.csv.gz")

# OUTPUT
opt_short_file = os.path.join(PROCESSED_DIR, "option_prices_earnings_short.parquet")

# =============================================================================
# Helper
# =============================================================================
def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))

# =============================================================================
# STEP 0: sanity check raw option file exists
# =============================================================================
print("\nRaw option file exists:", os.path.exists(raw_opt_file))
print("Raw option file path:", raw_opt_file)

# =============================================================================
# STEP 1: Load clean earnings event sample
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_prices_clean AS
SELECT *
FROM read_parquet('{earnings_prices_clean_file}')
""")

# =============================================================================
# STEP 2: Rebuild event -> secid mapping
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp_id_history AS
SELECT DISTINCT
    permno,
    date,
    NCUSIP AS ncusip,
    TICKER AS crsp_ticker
FROM read_parquet('{crsp_file}')
WHERE NCUSIP IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_crsp_id AS
SELECT
    e.*,
    c.ncusip,
    c.crsp_ticker
FROM earnings_prices_clean e
LEFT JOIN crsp_id_history c
  ON e.permno = c.permno
 AND e.prev_date = c.date
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_name_history AS
SELECT
    secid,
    ticker AS om_ticker,
    cusip AS om_cusip,
    effect_date
FROM read_parquet('{name_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_secid AS
SELECT *
FROM (
    SELECT
        e.*,
        o.secid,
        o.om_ticker,
        o.om_cusip,
        o.effect_date,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY o.effect_date DESC, o.secid
        ) AS rn
    FROM event_crsp_id e
    LEFT JOIN om_name_history o
      ON e.ncusip = o.om_cusip
     AND o.effect_date <= e.prev_date
) t
WHERE rn = 1
  AND secid IS NOT NULL
""")

# =============================================================================
# STEP 3: Keep only event rows that also have OM underlying price on prev_date
# =============================================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_sec_prices AS
SELECT
    secid,
    date,
    close AS om_close
FROM read_parquet('{sec_price_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_ready AS
SELECT
    e.*
FROM event_secid e
JOIN om_sec_prices s
  ON e.secid = s.secid
 AND e.prev_date = s.date
WHERE s.om_close IS NOT NULL
""")

# =============================================================================
# STEP 4: Build the exact secid-date pairs we need from raw options
# =============================================================================
con.execute("""
CREATE OR REPLACE TEMP TABLE needed_secid_dates AS
SELECT DISTINCT
    secid,
    prev_date AS trade_date
FROM event_ready
""")

# Optional: secid-only list
con.execute("""
CREATE OR REPLACE TEMP TABLE needed_secids AS
SELECT DISTINCT secid
FROM needed_secid_dates
""")

show(
    "1. Event-ready row count",
    "SELECT COUNT(*) AS n_rows FROM event_ready"
)

show(
    "2. Distinct secids needed",
    "SELECT COUNT(DISTINCT secid) AS n_secids FROM needed_secids"
)

show(
    "3. Distinct secid-trade_date pairs needed",
    "SELECT COUNT(*) AS n_secid_dates FROM needed_secid_dates"
)

# =============================================================================
# STEP 5: Inspect raw option schema first
# =============================================================================
show(
    "4. Raw option file schema",
    f"DESCRIBE SELECT * FROM read_csv_auto('{raw_opt_file}', compression='gzip')",
    n=200
)

# =============================================================================
# STEP 6: Build targeted short-DTE option subset from raw file
#
# Assumptions from your earlier file:
# - fields include secid, date, exdate, cp_flag, strike_price,
#   best_bid, best_offer, impl_volatility, delta, gamma, vega, theta,
#   volume, open_interest, optionid, ticker, cusip, exercise_style
#
# We now keep DTE between 0 and 21 instead of 7 to 119.
# =============================================================================
con.execute(f"""
COPY (
    SELECT
        TRY_CAST(secid AS BIGINT) AS secid,
        CAST(date AS DATE) AS date,
        CAST(exdate AS DATE) AS exdate,
        cp_flag,
        TRY_CAST(strike_price AS BIGINT) AS strike_price,
        TRY_CAST(best_bid AS DOUBLE) AS best_bid,
        TRY_CAST(best_offer AS DOUBLE) AS best_offer,
        (TRY_CAST(best_bid AS DOUBLE) + TRY_CAST(best_offer AS DOUBLE)) / 2.0 AS mid,
        TRY_CAST(impl_volatility AS DOUBLE) AS impl_volatility,
        TRY_CAST(delta AS DOUBLE) AS delta,
        TRY_CAST(gamma AS DOUBLE) AS gamma,
        TRY_CAST(vega AS DOUBLE) AS vega,
        TRY_CAST(theta AS DOUBLE) AS theta,
        TRY_CAST(volume AS BIGINT) AS volume,
        TRY_CAST(open_interest AS BIGINT) AS open_interest,
        TRY_CAST(optionid AS BIGINT) AS optionid,
        ticker,
        cusip,
        exercise_style,
        CAST(exdate AS DATE) - CAST(date AS DATE) AS dte
    FROM read_csv_auto(
        '{raw_opt_file}',
        compression='gzip',
        all_varchar=true
    )
    WHERE TRY_CAST(secid AS BIGINT) IN (SELECT secid FROM needed_secids)
      AND (TRY_CAST(secid AS BIGINT), CAST(date AS DATE)) IN (
            SELECT secid, trade_date
            FROM needed_secid_dates
      )
      AND CAST(exdate AS DATE) >= CAST(date AS DATE)
      AND (CAST(exdate AS DATE) - CAST(date AS DATE)) BETWEEN 0 AND 21
      AND cp_flag IN ('C', 'P')
      AND TRY_CAST(best_bid AS DOUBLE) IS NOT NULL
      AND TRY_CAST(best_offer AS DOUBLE) IS NOT NULL
      AND TRY_CAST(best_bid AS DOUBLE) >= 0
      AND TRY_CAST(best_offer AS DOUBLE) >= 0
) TO '{opt_short_file}' (FORMAT PARQUET)
""")

print(f"\nSaved targeted short-DTE options file to:\n{opt_short_file}")

# =============================================================================
# STEP 7: Diagnostics
# =============================================================================
show(
    "5. Row count in targeted short-DTE options file",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{opt_short_file}')"
)

show(
    "6. Distinct secids covered",
    f"SELECT COUNT(DISTINCT secid) AS n_secids FROM read_parquet('{opt_short_file}')"
)

show(
    "7. Date coverage",
    f"""
    SELECT
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        MIN(exdate) AS min_exdate,
        MAX(exdate) AS max_exdate
    FROM read_parquet('{opt_short_file}')
    """
)

show(
    "8. DTE distribution",
    f"""
    SELECT
        MIN(dte) AS min_dte,
        APPROX_QUANTILE(dte, 0.01) AS p01_dte,
        APPROX_QUANTILE(dte, 0.10) AS p10_dte,
        APPROX_QUANTILE(dte, 0.50) AS median_dte,
        APPROX_QUANTILE(dte, 0.90) AS p90_dte,
        APPROX_QUANTILE(dte, 0.99) AS p99_dte,
        MAX(dte) AS max_dte
    FROM read_parquet('{opt_short_file}')
    """
)

show(
    "9. Do we now have expiries on or before rdq for sample events?",
    f"""
    WITH option_chain_dates AS (
        SELECT DISTINCT secid, date, exdate
        FROM read_parquet('{opt_short_file}')
    )
    SELECT
        SUM(CASE WHEN has_before = 1 THEN 1 ELSE 0 END) AS events_with_before_expiry,
        SUM(CASE WHEN has_after = 1 THEN 1 ELSE 0 END) AS events_with_after_expiry,
        COUNT(*) AS total_events_checked
    FROM (
        SELECT
            e.gvkey,
            e.rdq,
            MAX(CASE WHEN c.exdate <= e.rdq THEN 1 ELSE 0 END) AS has_before,
            MAX(CASE WHEN c.exdate >  e.rdq THEN 1 ELSE 0 END) AS has_after
        FROM event_ready e
        LEFT JOIN option_chain_dates c
          ON e.secid = c.secid
         AND e.prev_date = c.date
        GROUP BY e.gvkey, e.rdq
    ) x
    """
)

show(
    "10. Sample rows from targeted short-DTE options file",
    f"""
    SELECT
        secid, date, exdate, cp_flag, strike_price, strike_price/1000.0 AS strike,
        best_bid, best_offer, mid, open_interest, volume, dte, ticker, cusip
    FROM read_parquet('{opt_short_file}')
    LIMIT 30
    """
)


Raw option file exists: True
Raw option file path: data\option_prices.csv.gz

1. Event-ready row count
 n_rows
  12417

2. Distinct secids needed
 n_secids
      483

3. Distinct secid-trade_date pairs needed
 n_secid_dates
         12417

4. Raw option file schema
     column_name column_type null  key default extra
           secid      BIGINT  YES None    None  None
            date        DATE  YES None    None  None
          symbol     VARCHAR  YES None    None  None
     symbol_flag      BIGINT  YES None    None  None
          exdate        DATE  YES None    None  None
       last_date        DATE  YES None    None  None
         cp_flag     VARCHAR  YES None    None  None
    strike_price      BIGINT  YES None    None  None
        best_bid      DOUBLE  YES None    None  None
      best_offer      DOUBLE  YES None    None  None
          volume      BIGINT  YES None    None  None
   open_interest      BIGINT  YES None    None  None
 impl_volatility      DOUBLE  YES None    No

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Saved targeted short-DTE options file to:
data\processed\option_prices_earnings_short.parquet

5. Row count in targeted short-DTE options file
 n_rows
1429763

6. Distinct secids covered
 n_secids
      475

7. Date coverage
  min_date   max_date min_exdate max_exdate
2018-02-07 2024-12-19 2018-02-16 2025-01-03

8. DTE distribution
 min_dte  p01_dte  p10_dte  median_dte  p90_dte  p99_dte  max_dte
       6        7        9          14       18       21       21

9. Do we now have expiries on or before rdq for sample events?
 events_with_before_expiry  events_with_after_expiry  total_events_checked
                       0.0                    9275.0                 12417

10. Sample rows from targeted short-DTE options file
 secid       date     exdate cp_flag  strike_price  strike  best_bid  best_offer    mid  open_interest  volume  dte ticker    cusip
101062 2018-03-14 2018-03-23       C        130000   130.0     86.15       90.60 88.375              0       0    9   ADBE 00724F10
1